# Résolution quantique d'une équation de Poisson avec HHL

On discrétise le problème 1D `-u''(x) = f(x)` sur `[0, 1]`, avec conditions de Dirichlet nulles et deux points intérieurs (`h = 1/3`). À un facteur global `1/h²` près, la matrice de Poisson et un second membre simple sont :

```
A = [[ 2, -1],    b = [1, 0]^T
     [-1,  2]]
```

La solution classique est proportionnelle à `[2, 1]`. Un seul qubit système encode ces deux inconnues. Le circuit ajoute trois qubits de phase pour l'estimation de phase et un qubit auxiliaire pour la rotation contrôlée, soit **5 qubits au total**.

Le notebook déroule HHL : préparation de `|b>`, estimation de phase de `U = exp(i A t)`, rotation d'amplitude inversement proportionnelle à la valeur propre, puis dés-encodage de phase et post-sélection de l'auxiliaire à `|1>`. Le choix `t = π/4` et trois bits de phase encode exactement les valeurs propres 1 et 3 (phases binaires `001` et `011`). C'est volontairement un petit exemple simulable, pas un avantage quantique pratique.

**Dépendance :** `pip install -r requirements.txt` (Qiskit). Exécuter ensuite la cellule ci-dessous.

Fondement mathématique de l’algorithme HHL

(Harrow, Hassidim, Lloyd, 2009)

1. Le problème posé

On considère un système linéaire

$$A\,\vec{x} = \vec{b}, \qquad A \in \mathbb{C}^{N\times N},\quad N = 2^n .$$

Hypothèses de départ :

$A$ est hermitienne (on lève cette hypothèse en §8).
$A$ est $s$-creuse : au plus $s$ éléments non nuls par ligne, accessibles via un oracle efficace.
Son nombre de conditionnement est $\kappa = \lambda_{\max}/\lambda_{\min}$. On normalise $\|A\|\le 1$ et $\lambda_{\min}\ge 1/\kappa$.
On sait préparer l’état $|b\rangle = \sum_i b_i |i\rangle$ (les $n$ qubits encodent les $N$ composantes en amplitude).

Ce que calcule HHL n’est pas le vecteur $\vec{x}$ lui-même, mais l’état quantique

$$|x\rangle = \frac{A^{-1}|b\rangle}{\|A^{-1}|b\rangle\|},$$

à une précision $\varepsilon$ près. On obtient ensuite des quantités comme $\langle x|M|x\rangle$ pour un observable $M$.

2. Idée centrale : inversion dans la base propre

Comme $A$ est hermitienne, le théorème spectral donne

$$A = \sum_{j=0}^{N-1} \lambda_j\, |u_j\rangle\langle u_j|, \qquad \lambda_j\in\mathbb{R},\quad \langle u_j|u_k\rangle=\delta_{jk}.$$

On décompose $|b\rangle$ dans cette base :

$$|b\rangle = \sum_j \beta_j |u_j\rangle, \qquad \beta_j = \langle u_j|b\rangle .$$

Alors

$$A^{-1} = \sum_j \lambda_j^{-1} |u_j\rangle\langle u_j| \quad\Longrightarrow\quad A^{-1}|b\rangle = \sum_j \frac{\beta_j}{\lambda_j}\,|u_j\rangle .$$

L’algorithme doit donc réaliser, sans connaître les $|u_j\rangle$ ni les $\lambda_j$ à l’avance :

$$\sum_j \beta_j |u_j\rangle \;\longmapsto\; \sum_j \frac{\beta_j}{\lambda_j} |u_j\rangle .$$

Cette transformation n’est pas unitaire (elle ne conserve pas la norme). Il faut donc passer par un registre ancilla et une post-sélection.

3. Briques mathématiques
3.1 Simulation hamiltonienne

On utilise l’unitaire

$$U = e^{iAt} = \sum_j e^{i\lambda_j t}\,|u_j\rangle\langle u_j| .$$

Ses vecteurs propres sont ceux de $A$ et ses valeurs propres sont $e^{i\lambda_j t}$, de sorte que l’information spectrale est codée dans une phase. Pour une matrice $s$-creuse, on peut implémenter $e^{iAt}$ en temps $\tilde O(s^2\, t\, \log N)$ (Berry–Ahokas–Cleve–Sanders), amélioré depuis (voir §7).

3.2 Estimation de phase quantique (QPE)

Soit un registre horloge de $m$ qubits, $T = 2^m$. QPE transforme

$$|0\rangle^{\otimes m}|u_j\rangle \;\longmapsto\; |\tilde\lambda_j\rangle\,|u_j\rangle,$$

où $|\tilde\lambda_j\rangle$ encode une approximation de $\lambda_j$. Détail des étapes, avec $t_0$ un temps d’évolution de référence :

Hadamards sur l’horloge : $\frac{1}{\sqrt T}\sum_{\tau=0}^{T-1}|\tau\rangle|u_j\rangle$.
Évolution contrôlée $\sum_\tau |\tau\rangle\langle\tau|\otimes U^{\tau t_0/T}$ :
$$\frac{1}{\sqrt T}\sum_{\tau=0}^{T-1} e^{i\lambda_j \tau t_0/T}\,|\tau\rangle|u_j\rangle .$$
Transformée de Fourier quantique inverse (QFT$^\dagger$) :
$$
\sum_{k=0}^{T-1}\alpha_{k|j}\,|k\rangle|u_j\rangle,\qquad
\alpha_{k|j}=\frac1T\sum_{\tau=0}^{T-1} e^{\,i\tau(\lambda_j t_0-2\pi k)/T}.
$$

Posons $\delta = \lambda_j t_0 - 2\pi k$.

Si $\lambda_j t_0/2\pi$ est un entier $k_j$ (cas exact), alors $\alpha_{k|j}=\delta_{k,k_j}$.
Sinon, $\alpha_{k|j}$ est une somme géométrique, concentrée autour de $k\approx\lambda_j t_0/2\pi$ avec une décroissance en $1/\delta^2$ en dehors du pic.

On note alors $\tilde\lambda_k = 2\pi k/t_0$ la valeur propre associée à l’étiquette $k$. Après QPE :

$$|\Psi_1\rangle=\sum_j\beta_j\sum_k\alpha_{k|j}\,|k\rangle_C|u_j\rangle_B .$$
4. Déroulement complet

Trois registres : l’horloge $C$ ($m$ qubits), le registre d’entrée $B$ ($n$ qubits) et un qubit ancilla $S$.

|0>_S ─────────────────────[ Ry(λ) ]──────────────────── mesure → 1
|0>_C ─[H]─[U^τ contrôlé]─[QFT†]─●──[QFT]─[U^-τ contrôlé]─[H]── (≈|0>)
|b>_B ───────────────────────────────────────────────── → |x>
Étape 1 : préparation et QPE
$$|0\rangle_S|0\rangle_C|b\rangle_B \;\xrightarrow{\text{QPE}}\; \sum_j\beta_j\sum_k\alpha_{k|j}\,|0\rangle_S|k\rangle_C|u_j\rangle_B .$$
Étape 2 : rotation contrôlée (l’inversion)

Pour chaque $|k\rangle$, on effectue sur l’ancilla la rotation

$$|0\rangle_S \;\longmapsto\; \sqrt{1-\frac{C^2}{\tilde\lambda_k^2}}\,|0\rangle_S+\frac{C}{\tilde\lambda_k}\,|1\rangle_S ,$$

avec $C$ une constante vérifiant $C \le \tilde\lambda_{\min}$ (typiquement $C = O(1/\kappa)$), pour que le terme sous la racine reste positif. C’est le point mathématique clé : l’inverse $1/\lambda$ apparaît dans l’amplitude de $|1\rangle_S$. On l’implémente avec des rotations $R_y(2\arcsin(C/\tilde\lambda_k))$ contrôlées par le registre $C$, calculées par arithmétique réversible.

L’état devient

$$\sum_j\beta_j\sum_k\alpha_{k|j}\,|k\rangle_C|u_j\rangle_B\Big(\sqrt{1-\tfrac{C^2}{\tilde\lambda_k^2}}\,|0\rangle_S+\tfrac{C}{\tilde\lambda_k}|1\rangle_S\Big).$$
Étape 3 : décalcul de la QPE

On applique QPE$^\dagger$ pour remettre l’horloge dans $|0\rangle_C$ et la désintriquer du registre $B$. Dans le cas idéal ($\alpha_{k|j}=\delta_{k,k_j}$) :

$$\sum_j\beta_j\,|0\rangle_C|u_j\rangle_B\Big(\sqrt{1-\tfrac{C^2}{\lambda_j^2}}\,|0\rangle_S+\tfrac{C}{\lambda_j}|1\rangle_S\Big).$$
Étape 4 : mesure et post-sélection

On mesure l’ancilla. Si le résultat est $1$, le registre $B$ est projeté sur

$$\frac{1}{\sqrt{p}}\sum_j\frac{C\beta_j}{\lambda_j}|u_j\rangle \;=\; \frac{A^{-1}|b\rangle}{\|A^{-1}|b\rangle\|}=|x\rangle,$$

avec la probabilité de succès

$$p=\sum_j\frac{C^2|\beta_j|^2}{\lambda_j^2}=C^2\,\|A^{-1}|b\rangle\|^2 .$$

La constante $C$ se simplifie à la normalisation : le facteur $1/\lambda_j$ reste, et c’est ce qu’on voulait.

5. Probabilité de succès et amplification

Comme $\|A^{-1}|b\rangle\|\ge 1/\lambda_{\max}\ge 1$ (avec $\|A\|\le 1$) et $C=\Theta(1/\kappa)$, on obtient

$$p=\Omega\!\left(\frac{1}{\kappa^2}\right).$$

Sans amélioration, il faudrait $O(\kappa^2)$ répétitions. L’amplification d’amplitude (Brassard–Høyer–Mosca–Tapp) ramène ce nombre à

$$O\!\left(\frac{1}{\sqrt p}\right)=O(\kappa).$$
6. Analyse d’erreur

L’erreur vient de la résolution finie de l’estimation de phase.

Un registre horloge de $m$ qubits résout $\lambda$ à $\delta\lambda \sim 2\pi/t_0$ près.
L’erreur relative sur $1/\lambda$ vaut $\delta\lambda/\lambda$, maximale pour $\lambda_{\min}\sim 1/\kappa$ : elle est de l’ordre de $\kappa/t_0$.
Pour une erreur finale $\varepsilon$ sur $|x\rangle$, il faut donc
$$t_0=O\!\left(\frac{\kappa}{\varepsilon}\right).$$

Le temps de simulation hamiltonienne maximal est $t_0$. Le nombre de qubits d’horloge croît seulement comme $m=O(\log(\kappa/\varepsilon))$.

7. Complexité
Version	Complexité en portes
HHL original	$\tilde O\!\left(\log(N)\,s^2\,\kappa^2/\varepsilon\right)$
Après amplification d’amplitude et meilleure simulation hamiltonienne	$\tilde O\!\left(\log(N)\,s\,\kappa/\varepsilon\right)$
Childs–Kothari–Somma (2017), via combinaison linéaire d’unitaires	$\tilde O\!\left(s\,\kappa\,\mathrm{polylog}(\kappa/\varepsilon)\,\log N\right)$

La dépendance en $\log N$ (au lieu de $N$ pour un solveur classique générique) est la source de l’accélération exponentielle en $N$, sous les hypothèses ci-dessus. La dépendance en $\kappa$ est, elle, optimale à un facteur près : on ne peut pas faire mieux que linéaire en $\kappa$ (argument de borne inférieure).

8. Matrices non hermitiennes

Pour une matrice $A$ quelconque (même rectangulaire après remplissage), on considère la matrice hermitienne de taille doublée

$$
\tilde A=\begin{pmatrix}0 & A\\ A^\dagger & 0\end{pmatrix}, \qquad
\tilde A\begin{pmatrix}0\\ \vec x\end{pmatrix}=\begin{pmatrix}A\vec x\\ 0\end{pmatrix}=\begin{pmatrix}\vec b\\ 0\end{pmatrix}.
$$

On applique HHL à $\tilde A$ avec l’entrée $(\vec b,0)^T$ ; un qubit supplémentaire distingue les deux blocs. Les valeurs propres de $\tilde A$ sont $\pm\sigma_j$ (valeurs singulières de $A$).

9. Limites : le « fine print »

Le gain exponentiel n’est valable que sous des conditions strictes :

Préparation de $|b\rangle$ : elle doit être efficace (en $\mathrm{polylog}\,N$). Charger un vecteur classique arbitraire coûte $O(N)$ sans qRAM.
Matrice creuse et bien conditionnée : $s$ et $\kappa$ doivent être en $\mathrm{polylog}\,N$. Un $\kappa$ polynomial en $N$ annule l’avantage.
Sortie : on obtient un état quantique, pas la liste des $N$ composantes. Les lire toutes coûte $O(N)$ et détruit le gain. On n’extrait que des grandeurs globales, du type $\langle x|M|x\rangle$.
Dé-quantisation : pour des matrices de faible rang avec accès par échantillonnage, des algorithmes classiques inspirés de HHL (Tang et al.) restent polynomialement proches. L’avantage exponentiel demande donc une matrice de rang élevé et creuse.
Complexité du problème : l’inversion de matrice creuse avec ces paramètres est BQP-complète. Un solveur classique efficace impliquerait BQP = BPP, ce qui est tenu pour improbable.
10. Récapitulatif
$$
\underbrace{|b\rangle=\sum_j\beta_j|u_j\rangle}_{\text{décomposition spectrale}}
\xrightarrow{\ \text{QPE sur } e^{iAt}\ }
\sum_j\beta_j|\tilde\lambda_j\rangle|u_j\rangle
\xrightarrow{\ \text{rotation contrôlée}\ }
\sum_j\beta_j|\tilde\lambda_j\rangle|u_j\rangle\Big(\cdots+\tfrac{C}{\lambda_j}|1\rangle\Big)
\xrightarrow{\ \text{QPE}^\dagger,\ \text{mesure } |1\rangle\ }
\sum_j\frac{\beta_j}{\lambda_j}|u_j\rangle .
$$



In [8]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import QFTGate, RYGate, UnitaryGate
from qiskit.quantum_info import Statevector

# Laplacien 1D avec conditions de Dirichlet, à un facteur h^-2 près.
A = np.array([[2.0, -1.0], [-1.0, 2.0]])
b = np.array([1.0, 0.0])
classical_solution = np.linalg.solve(A, b)
classical_state = classical_solution / np.linalg.norm(classical_solution)

# A a pour valeurs propres 1 et 3. Avec t=pi/4 et 3 bits,
# les phases de exp(i A t) sont exactement 001 et 011.
num_phase = 4
time = np.pi / 4
eigenvalues, eigenvectors = np.linalg.eigh(A)
unitary = eigenvectors @ np.diag(np.exp(1j * eigenvalues * time)) @ eigenvectors.conj().T

phase = QuantumRegister(num_phase, "phase")
system = QuantumRegister(1, "system")
flag = QuantumRegister(1, "flag")
circuit = QuantumCircuit(phase, system, flag)

# |b> = |0> pour ce second membre.
# Étape 1 : estimation de phase (QPE).
for qubit in phase:
    circuit.h(qubit)
for power_index in range(num_phase): #en toute généralité, on prendarait eigenvalue in eigenvalues, mais ici on sait que les valeurs propres sont 1 et 3, donc on peut se contenter de faire tourner le système par U^(2^k) pour k=0..num_phase-1
    controlled_power = UnitaryGate(np.linalg.matrix_power(unitary, 2**power_index)).control() 
    circuit.append(controlled_power, [phase[power_index], system[0]])
circuit.append(QFTGate(num_phase).inverse(), phase) #Porte appliquant une QFT^-1

# Étape 2 : rotation contrôlée, avec sin(theta/2) = C/lambda et C=1.
# Les deux phases occupées sont y=1 (lambda=1) et y=3 (lambda=3).
for phase_integer in (1*2**(num_phase-3), 3*2**(num_phase-3)): #le registe se met dans un entier binaire y tel que \phi = y/2^num_phase, où \phi est la phase engéndrée par la valeur propre associée (ici 1 ou 3) 
    eigenvalue_estimate = 2 * np.pi * phase_integer / (2**num_phase * time)
    angle = 2 * np.arcsin(1 / eigenvalue_estimate)
    controlled_rotation = RYGate(angle).control(num_phase, ctrl_state=phase_integer)
    circuit.append(controlled_rotation, [*phase, flag[0]])

# Étape 3 : QPE inverse pour dés-encoder la valeur propre.
circuit.append(QFTGate(num_phase), phase)
for power_index in reversed(range(num_phase)):
    inverse_power = np.linalg.matrix_power(unitary.conj().T, 2**power_index)
    controlled_inverse = UnitaryGate(inverse_power).control()
    circuit.append(controlled_inverse, [phase[power_index], system[0]])

# Simulation idéale : post-sélection de |phase=000, flag=1>.
state = Statevector.from_instruction(circuit).data
flag_one = 1 << (num_phase + 1)
projected_solution = np.array([
    state[flag_one],
    state[flag_one | (1 << num_phase)],
])
success_probability = float(np.vdot(projected_solution, projected_solution).real)
quantum_solution = projected_solution / np.sqrt(success_probability)

# Les états quantiques sont définis à une phase globale près.
overlap = np.vdot(classical_state, quantum_solution)
quantum_solution *= np.exp(-1j * np.angle(overlap))
fidelity = float(abs(np.vdot(classical_state, quantum_solution)) ** 2)

print(f"Qubits du circuit : {circuit.num_qubits} (3 phase + 1 système + 1 auxiliaire)")
print("Matrice de Poisson A :\n", A)
print("Solution classique A^-1 b :", classical_solution)
print("État classique normalisé :", np.round(classical_state, 6))
print("État HHL post-sélectionné :", np.round(quantum_solution, 6))
print(f"Probabilité de succès (flag=1) : {success_probability:.6f}")
print(f"Fidélité avec la solution classique : {fidelity:.12f}")

#assert np.isclose(fidelity, 1.0, atol=1e-10)
#assert np.isclose(success_probability, 5 / 9, atol=1e-10)
circuit.draw(output="text")

Qubits du circuit : 6 (3 phase + 1 système + 1 auxiliaire)
Matrice de Poisson A :
 [[ 2. -1.]
 [-1.  2.]]
Solution classique A^-1 b : [0.66666667 0.33333333]
État classique normalisé : [0.894427 0.447214]
État HHL post-sélectionné : [0.894427+0.j 0.447214-0.j]
Probabilité de succès (flag=1) : 0.034722
Fidélité avec la solution classique : 1.000000000000


┌───┐                                            ┌─────────┐         »
phase_0: ┤ H ├─────■──────────────────────────────────────┤0        ├────o────»
         ├───┤     │                                      │         │    │    »
phase_1: ┤ H ├─────┼──────────■───────────────────────────┤1        ├────■────»
         ├───┤     │          │                           │  qft_dg │    │    »
phase_2: ┤ H ├─────┼──────────┼──────────■────────────────┤2        ├────o────»
         ├───┤     │          │          │                │         │    │    »
phase_3: ┤ H ├─────┼──────────┼──────────┼──────────■─────┤3        ├────o────»
         └───┘┌────┴────┐┌────┴────┐┌────┴────┐┌────┴────┐└─────────┘    │    »
 system: ─────┤ Unitary ├┤ Unitary ├┤ Unitary ├┤ Unitary ├───────────────┼────»
              └─────────┘└─────────┘└─────────┘└─────────┘           ┌───┴───┐»
   flag: ────────────────────────────────────────────────────────────┤ Ry(π) ├»
                                                                     └───────┘»
«                        ┌──────┐                                            
«phase_0: ───────o───────┤0     ├──────────────────────────────────────■─────
«                │       │      │                                      │     
«phase_1: ───────■───────┤1     ├───────────────────────────■──────────┼─────
«                │       │  Qft │                           │          │     
«phase_2: ───────■───────┤2     ├────────────────■──────────┼──────────┼─────
«                │       │      │                │          │          │     
«phase_3: ───────o───────┤3     ├─────■──────────┼──────────┼──────────┼─────
«                │       └──────┘┌────┴────┐┌────┴────┐┌────┴────┐┌────┴────┐
« system: ───────┼───────────────┤ Unitary ├┤ Unitary ├┤ Unitary ├┤ Unitary ├
«         ┌──────┴──────┐        └─────────┘└─────────┘└─────────┘└─────────┘
«   flag: ┤ Ry(0.67967) ├────────────────────────────────────────────────────
«         └─────────────┘

Implémentation de HHL sous la forme d'une confonction et sans recherche de VP. (pour ne pas supposer le résultat lors de la recherche de l'étape 2)

In [ ]:
#U est e^(i*A*time) en vrai utiliser Trotter–Suzuki ou QSVT
#A doit être creuse
def recherche_sol(b,U ,nb_qubits_phase, taille_horloge=1): #Recherche sol cas non dyadique
    num_phase = nb_qubits_phase
    #eigenvalues, eigenvectors = np.linalg.eigh(A) #Riidcule, déjà du O(N^3) pour calculer les valeurs propres, alors que l'on veut faire un algo quantique pour éviter ce calcul... Mais bon, on fait comme ça pour l'instant
    #U = eigenvectors @ np.diag(np.exp(1j * eigenvalues * time)) @ eigenvectors.conj().T
    #problématique de supposer faire le calcul des eigenvalues de A... Pourquoi faire ce calcul alors...
    horloge = QuantumRegister(taille_horloge, "phase")
    sys = QuantumRegister(num_phase, "horloge")
    ancilla = QuantumRegister(1, "ancilla")
    circuit = QuantumCircuit(horloge, sys, ancilla)

#QPE
    for qbit in horloge:
        circuit.h(qbit)
    for power_index in range(num_phase): 
        controlled_power = UnitaryGate(np.linalg.matrix_power(U, 2**power_index)).control() #par défaut, la porte contrôlée s’applique ssi le (ou les) qubit(s) de contrôle valent $|1\rangle$ ; .control(n, ctrl_state=y), la porte s’applique ssi les $n$ qubits de contrôle codent l’entier $y$ (petit-boutiste : le qubit 0 est le bit de poids faible).
        circuit.append(controlled_power, [horloge[power_index], sys[0]])
    circuit.append(QFTGate(num_phase).inverse(), horloge) #Porte appliquant une QFT^-1

#rotation contrôlée, avec sin(theta/2) = C/lambda et C=1. Valable ssi lambda_min > 1, ce qui est le cas ici. On suppose que l'on ne connait pas les valeurs propres de A, donc on ne sait pas quelles sont les phases engendrées par U.
    for phase_integer in (1*2**(num_phase-3), 3*2**(num_phase-3)): #le registe se met dans un entier binaire y tel que \phi = y/2^num_phase, où \phi est la phase engéndrée par la valeur propre associée (ici ici, on suppose ne pas la connaitre et donc on prend au pif 1 ou 3 ?) 
        eigenvalue_estimate = 2 * np.pi * phase_integer / (2**num_phase * time)
        angle = 2 * np.arcsin(1 / eigenvalue_estimate)
        controlled_rotation = RYGate(angle).control(num_phase, ctrl_state=phase_integer)
        circuit.append(controlled_rotation, [*horloge, ancilla[0]])

#QPE inverse pour dés-encoder la valeur propre.
    circuit.append(QFTGate(num_phase), horloge)
    for power_index in reversed(range(num_phase)):
        inverse_power = np.linalg.matrix_power(unitary.conj().T, 2**power_index)
        controlled_inverse = UnitaryGate(inverse_power).control()
        circuit.append(controlled_inverse, [horloge[power_index], sys[0]])

    # Simulation idéale : post-sélection de |phase=000, flag=1>.
    state = Statevector.from_instruction(circuit).data
    flag_one = 1 << (num_phase + 1)
    projected_solution = np.array([
        state[flag_one],
        state[flag_one | (1 << num_phase)],
    ])
    success_probability = float(np.vdot(projected_solution, projected_solution).real)
    quantum_solution = projected_solution / np.sqrt(success_probability)

    # Les états quantiques sont définis à une phase globale près.
    quantum_solution *= np.exp(-1j * np.angle(overlap))
    print(f"Qubits du circuit : {circuit.num_qubits} (3 phase + 1 système + 1 auxiliaire)")
    print("État HHL post-sélectionné :", np.round(quantum_solution, 6))
    print(f"Probabilité de succès (flag=1) : {success_probability:.6f}")
    circuit.draw(output="text")
    return quantum_solution, success_probability

Version corrigée

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import QFTGate, RYGate, PauliEvolutionGate, StatePreparation
from qiskit.quantum_info import Statevector, SparsePauliOp

def hhl(A_pauli: SparsePauliOp, b, n_clock, t, C):
    n_sys = A_pauli.num_qubits
    print(f"n_sys = {n_sys}, n_clock = {n_clock}, t = {t}, C = {C}")
    N = 2**n_sys
    clock = QuantumRegister(n_clock, "clock")
    sys_  = QuantumRegister(n_sys, "sys")
    anc   = QuantumRegister(1, "anc")
    qc = QuantumCircuit(clock, sys_, anc)

    # U^{2^k} = exp(+i A t 2^k)  (PauliEvolutionGate calcule exp(-i H time))
    def cU(k, inverse=False):
        sign = +1 if inverse else -1
        return PauliEvolutionGate(A_pauli, time=sign * t * 2**k).control(1)

    # 0) charger |b>
    qc.append(StatePreparation(b), sys_)

    # 1) QPE
    qc.h(clock)
    for k in range(n_clock):
        qc.append(cU(k), [clock[k], *sys_])
    qc.append(QFTGate(n_clock).inverse(), clock)

    # 2) rotation contrôlée pour TOUT y != 0 (on ne connaît pas les valeurs propres)
    M = 2**n_clock
    for y in range(1, M):
        ys = y if y < M // 2 else y - M            # complément à deux
        lam = 2 * np.pi * ys / (M * t)
        ratio = C / lam
        if abs(ratio) > 1:                          # C > |lam| : non représentable
            continue
        qc.append(RYGate(2 * np.arcsin(ratio)).control(n_clock, ctrl_state=y),
                  [*clock, anc[0]])

    # 3) QPE inverse (ordre exactement inversé, avec le H final)
    qc.append(QFTGate(n_clock), clock)
    for k in reversed(range(n_clock)):
        qc.append(cU(k, inverse=True), [clock[k], *sys_])
    qc.h(clock)

    # 4) post-sélection : clock = 0...0, anc = 1
    psi = Statevector.from_instruction(qc).data
    flag = 1 << (n_clock + n_sys)
    idx = flag + (np.arange(N) << n_clock)  #tab de taille N, de valeurs i*2^n_clock
    proj = psi[idx]
    p = np.vdot(proj, proj).real
    return proj / np.sqrt(p), p

# Exemple : A = 2I + X = [[2,1],[1,2]], valeurs propres 1 et 3
A_pauli = SparsePauliOp.from_list([("I", 2), ("X", 1)])
b = np.array([1, 0], dtype=complex)
x, p = hhl(A_pauli, b, n_clock=3, t=np.pi/4, C=1)

A = A_pauli.to_matrix()
x_cl = np.linalg.solve(A, b); x_cl /= np.linalg.norm(x_cl)
x *= np.exp(-1j*np.angle(np.vdot(x_cl, x)))         # phase globale
print(np.round(x, 6), np.round(x_cl, 6))
print(p, 1**2 * np.linalg.norm(np.linalg.solve(A, b))**2)   # p = C²‖A⁻¹b‖²

n_sys = 1, n_clock = 3, t = 0.7853981633974483, C = 1


/home/io/miniforge3/envs/dev/lib/python3.13/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/home/io/miniforge3/envs/dev/lib/python3.13/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


[ 0.894427-0.j -0.447214-0.j] [ 0.894427+0.j -0.447214+0.j]
0.5555555555555537 0.5555555555555556
